In [95]:
import pandas as pd
import numpy as np
import scipy
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn
import torch    
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import regex as re  

In [23]:
df_test = pd.read_csv('datasets/banking77_test.csv')
df_train = pd.read_csv('datasets/banking77_train.csv')

print(df_test.head())
print(df_train.head())

                                                text      category
0                           How do I locate my card?  card_arrival
1  I still have not received my new card, I order...  card_arrival
2  I ordered a card but it has not arrived. Help ...  card_arrival
3   Is there a way to know when my card will arrive?  card_arrival
4                       My card has not arrived yet.  card_arrival
                                                text      category
0                     I am still waiting on my card?  card_arrival
1  What can I do if my card still hasn't arrived ...  card_arrival
2  I have been waiting over a week. Is the card s...  card_arrival
3  Can I track my card while it is in the process...  card_arrival
4  How do I know if I will get my card, or if it ...  card_arrival


In [ ]:
pd.isnull(df_test).sum(
)
pd.isnull(df_train).sum(
)

print(df_test.duplicated().sum())
print(df_train.duplicated().sum())

In [ ]:
df_test.dtypes
df_train.dtypes

In [26]:
df_test['category'] = df_test['category'].astype('category')
df_train['category'] = df_train['category'].astype('category')

In [30]:
df_test.describe(include='all')
df_train.describe(include='all')

,text,category
count,10003,10003
unique,10003,77
top,I am still waiting on my card?,card_payment_fee_charged
freq,1,187


In [54]:
print("Label distribution in the training set:")
print(df_train['category'].value_counts())
label_pcts = df_train['category'].value_counts(normalize=True) * 100
print(label_pcts)

Label distribution in the training set:
category
card_payment_fee_charged                            187
direct_debit_payment_not_recognised                 182
balance_not_updated_after_cheque_or_cash_deposit    181
wrong_amount_of_cash_received                       180
cash_withdrawal_charge                              177
                                                   ... 
lost_or_stolen_card                                  82
card_swallowed                                       61
card_acceptance                                      59
virtual_card_not_working                             41
contactless_not_working                              35
Name: count, Length: 77, dtype: int64
category
card_payment_fee_charged                            1.869439
direct_debit_payment_not_recognised                 1.819454
balance_not_updated_after_cheque_or_cash_deposit    1.809457
wrong_amount_of_cash_received                       1.799460
cash_withdrawal_charge                      

In [63]:
df_train['Query length'] = df_train['text'].astype(str).apply(lambda x: len(x.split()))
length_stats = df_train.groupby('category')['Query length'].describe().reset_index()
print(length_stats)

                                   category  count       mean        std  min  \
0                     Refund_not_showing_up  162.0  15.808642  12.293782  4.0   
1                          activate_my_card  159.0   9.150943   3.349265  3.0   
2                                 age_limit  110.0   9.236364   2.785702  4.0   
3                   apple_pay_or_google_pay  126.0  10.341270   3.575836  5.0   
4                               atm_support   87.0   7.689655   2.001804  4.0   
..                                      ...    ...        ...        ...  ...   
72                 virtual_card_not_working   41.0   9.365854   4.575785  5.0   
73                       visa_or_mastercard  135.0   8.155556   2.617013  3.0   
74                      why_verify_identity  121.0   9.322314   3.165478  4.0   
75            wrong_amount_of_cash_received  180.0  15.283333   9.658571  4.0   
76  wrong_exchange_rate_for_cash_withdrawal  163.0  15.460123   8.351274  5.0   

     25%   50%   75%   max 

/var/folders/r8/sx398knx0jvg49_cwdb_pz1m0000gn/T/ipykernel_9866/1015264019.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  length_stats = df_train.groupby('category')['Query length'].describe().reset_index()


In [64]:
df_test['Query length'] = df_test['text'].astype(str).apply(lambda x: len(x.split()))

length_stats = df_test.groupby('category')['Query length'].describe().reset_index()
print(length_stats)

                                   category  count    mean        std  min  \
0                     Refund_not_showing_up   40.0  15.575  11.265758  4.0   
1                          activate_my_card   40.0   9.250   3.372209  3.0   
2                                 age_limit   40.0  10.000   2.736271  5.0   
3                   apple_pay_or_google_pay   40.0  10.475   3.522728  4.0   
4                               atm_support   40.0   7.475   1.999840  4.0   
..                                      ...    ...     ...        ...  ...   
72                 virtual_card_not_working   40.0  10.475   4.224395  5.0   
73                       visa_or_mastercard   40.0   7.775   2.369545  4.0   
74                      why_verify_identity   40.0   9.325   2.786299  5.0   
75            wrong_amount_of_cash_received   40.0  15.225   8.748590  6.0   
76  wrong_exchange_rate_for_cash_withdrawal   40.0  16.400   8.267328  7.0   

      25%   50%    75%   max  
0    8.00  10.5  17.25  45.0  
1

/var/folders/r8/sx398knx0jvg49_cwdb_pz1m0000gn/T/ipykernel_9866/790467920.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  length_stats = df_test.groupby('category')['Query length'].describe().reset_index()


In [ ]:
# Find most commonly used words in the training set

def split_into_words(text):
    lowercase_text = text.lower()
    split_words = re.split("\W+", lowercase_text)
    return split_words

number_of_words = df_train['text'].astype(str).apply(lambda x: len(x.split()))  

stopwords = ['a', 'are', 'am', 'an', 'the', 'my', 'you', 'it', 'and', 'to', 'me', 'has', 'had', 'will', 'was', 'on', 'of', 'be', 'from', 'was', 'have', 'in', 'for', 'is', 'can', 'do', 'i', 'or', 'but', 'if', 'then', 'else', 'when', 'where', 'why', 'how', 'what', 'which', 'who', 'whom', 'whose', 'this', 'that', 'these', 'those']

full_text = df_train['text'].str.lower().str.cat(sep=' ')

clean_text = re.sub(r'[!"\$%&\'()*+,\-.\/:;=#@?\[\\\]^_`{|}~]', ' ', full_text)

meaningful_words = [word for word in clean_text.split() if word not in stopwords]
meaningful_words_count = Counter(meaningful_words)
most_frequent_meaningful_words = meaningful_words_count.most_common(30)

print(most_frequent_meaningful_words)


[('card', 2682), ('t', 1582), ('up', 1360), ('account', 1352), ('money', 1133), ('transfer', 1084), ('top', 1045), ('not', 967), ('there', 925), ('get', 808), ('payment', 751), ('need', 698), ('cash', 691), ('with', 608), ('exchange', 549), ('charged', 529), ('please', 515), ('s', 511), ('atm', 482), ('app', 469), ('fee', 458), ('been', 457), ('use', 427), ('pending', 424), ('did', 388), ('help', 369), ('long', 362), ('made', 356), ('transaction', 354), ('still', 351)]


In [96]:
# TF-IDF Vectorisation

label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(df_train['category'])
y_test = label_encoder.transform(df_test['category'])

# Save number of unique classes for MLP output layer
num_classes = len(label_encoder.classes_)
print(f"Total target intent classes: {num_classes}")


tfidf_vectoriser = TfidfVectorizer(
    stop_words='english', 
    ngram_range=(1,2), 
    max_features=5000, 
    token_pattern=r'\b\w\w+\b')
X_train_tfidf = tfidf_vectoriser.fit_transform(df_train['text'])
X_test_tfidf = tfidf_vectoriser.transform(df_test['text'])

print(f"X_train shape: {X_train_tfidf.shape}")
print(f"X_test shape: {X_test_tfidf.shape}")    



Total target intent classes: 77
X_train shape: (10003, 5000)
X_test shape: (3080, 5000)


In [97]:
# Print 10 random features captured by vectoriser
import random
feature_names = tfidf_vectoriser.get_feature_names_out()
print("Sample extracted features (including bigrams):")
print(random.sample(list(feature_names), 10))

Sample extracted features (including bigrams):
['money amex', 'just did', 'money country', 'atm keeps', 'charged getting', 'resolved', 'did card', 'suddenly declined', 'number disposable', 'hasn delivered']


In [103]:
# Prepare data arrays
# Convert sparse TF-IDF matrices to dense arrays for PyTorch

X_train_dense = X_train_tfidf.toarray().astype(np.float32)
X_test_dense = X_test_tfidf.toarray().astype(np.float32)

# Custom PyTorch Dataset to handle input features and label integer indices
class BankingIntentDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long) # CrossEntropy expects labels as long integers

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Instatitate datasets and dataloaders
train_dataset = BankingIntentDataset(X_train_dense, y_train)
test_dataset = BankingIntentDataset(X_test_dense, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [104]:
# Build and train MLP architecture
class IntentMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(IntentMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

input_dim = X_train_dense.shape[1]
hidden_dim = 128
output_dim = num_classes

model = IntentMLP(input_dim, hidden_dim, output_dim)

In [105]:
criterion = nn.CrossEntropyLoss()
optimiser = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

In [106]:
epochs = 10
print("Starting training")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for batch_x, batch_y in train_loader:
        optimiser.zero_grad() # Clear gradient tracking buffers

        # Forward pass
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        # Backward pass and parameter updates
        loss.backward()
        optimiser.step()

        # Track training metrics
        running_loss += loss.item() * batch_x.size(0)
        _, predicted = torch.max(outputs, 1)
        correct_predictions += (predicted == batch_y).sum().item()
        total_samples += batch_y.size(0)
    
    # Calculate performance metrics per epoch
    epoch_loss = running_loss / total_samples
    epoch_accuracy = (correct_predictions / total_samples) * 100
    print(f"Epoch {epoch+1:02d}/{epochs}, Loss average: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%")


Starting training
Epoch 01/10, Loss average: 4.0360, Accuracy: 32.51%
Epoch 02/10, Loss average: 2.3742, Accuracy: 70.52%
Epoch 03/10, Loss average: 1.1040, Accuracy: 83.80%
Epoch 04/10, Loss average: 0.6653, Accuracy: 88.76%
Epoch 05/10, Loss average: 0.4776, Accuracy: 90.80%
Epoch 06/10, Loss average: 0.3720, Accuracy: 92.58%
Epoch 07/10, Loss average: 0.3041, Accuracy: 93.71%
Epoch 08/10, Loss average: 0.2564, Accuracy: 94.64%
Epoch 09/10, Loss average: 0.2210, Accuracy: 95.28%
Epoch 10/10, Loss average: 0.1932, Accuracy: 95.84%


In [107]:
# Evaluate on test dataset
model.eval()
all_predictions = []
all_actuals = []

print("Evaluating on test dataset")

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x)
        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(predicted.numpy())
        all_actuals.extend(batch_y.numpy())

print(f"Overall test accuracy: {accuracy_score(all_actuals, all_predictions) * 100:.2f}%")
print(f"F1 score: {f1_score(all_actuals, all_predictions, average='macro'):.4f}")
print(f"Weighted F1 score: {f1_score(all_actuals, all_predictions, average='weighted'):.4f}")

print(classification_report(all_actuals, all_predictions, target_names=label_encoder.classes_))


Evaluating on test dataset
Overall test accuracy: 84.94%
F1 score: 0.8501
Weighted F1 score: 0.8501
                                                  precision    recall  f1-score   support

                           Refund_not_showing_up       0.91      0.97      0.94        40
                                activate_my_card       0.95      0.95      0.95        40
                                       age_limit       1.00      0.97      0.99        40
                         apple_pay_or_google_pay       1.00      1.00      1.00        40
                                     atm_support       0.95      0.95      0.95        40
                                automatic_top_up       1.00      0.95      0.97        40
         balance_not_updated_after_bank_transfer       0.62      0.80      0.70        40
balance_not_updated_after_cheque_or_cash_deposit       0.92      0.85      0.88        40
                         beneficiary_not_allowed       0.94      0.85      0.89        40